<a href="https://colab.research.google.com/github/saragava-crypto/FundamentosEngenhariaQuimica/blob/main/FCEQ_Prova_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from dataclasses import dataclass

@dataclass
class ParametrosBioreator:
  """
  Equação do Modelo:
  ------------------
  dsdt = -Vmax(T) * (S / (Km + S))

  Parâmetros do Modelo:
  --------------------
  S: float
    Concentração de Substrato [mg/L]
  T: float
    Tempo de Simulação [h]
  Vmax: float
    Velocidade Máxima de Consumo [mg/L.h]
  Km: float
    Constante de Michaelis-Menten [mg/L]
  """

T: float = 30.0
Km: float = 20.0
Vmax_ref:float = 5.0
Ea: float = 30000.0
R: float = 8.314
Tref: float = 298.15

def Vmax_Arrhenius(T, params):
  """
  A dependência da velocidade máxima com a temperatura é dada pela
  equação de Arrhenius, em que as temperaturas devem ser convertidas
  para Kelvin.

  Equação do Modelo Vmax(T)
  --------------------------
  Vmax(T) = Vmax,ref * exp((-Ea / R)*((1/T)-(1/Tref)))

  Parâmetros do Modelo Vmax(T)
  ----------------------------
  Vmax,ref: float
    Velocidade Máxima [mg/L.h]
  Ea: float
    Energia de Ativação [J/mol]
  R: float
    Constante dos Gases Ideias [J/mol.K]
  T: float
    Temperatura de Referência [K]
 """
  return params.Vmax_ref * np.exp((-params.Ea / (params.R)) * ((1/params.T) - (1/params.Tref)))

def Bioreator_modelo(t,S,T,params):
  dsdt = -Vmax_Arrhenius * (S / (params.Km + S))
  return dsdt

  #condição inicial
  S0 = 100.0

  #Intervalo de tempo em horas
  t_span = (0,params.T)

  sol = solve_ivp(Bioreator_modelo,t_span,S0,args=(params,))

  #extração dos resultados
  t = sol.t
  concentracao = sol.y[0]

def Simula_Bioreator(S0,params = None):
  if params is None:
    params = ParametrosBioreator()

  t_span = (0,params.T)
  sol = solve_ivp(Bioreator_modelo,t_span,S0,args=(params,))

  t = sol.t
  concentracao = sol.y[0]
  return t, concentracao

  plt.plot(t,concentracao)
  plt.title("Concentração x Tempo")
  plt.xlabel("Tempo [h]")
  plt.ylabel("Concentração [mg/L]")
  plt.legend()
  plt.grid(True)
  plt.show()
